In [1]:
%load_ext autoreload
%autoreload 2
import plotly.io as pio

from trader import StockDataLoader

pio.get_chrome()


period = "20y"

tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "JPM",
    "PG",
    "JNJ",
    "SPY",
    "QQQ",
    "XLE",
    "GLD",
    "TLT",
    "SHY",
]

df, train_env_data, val_env_data = StockDataLoader(
    tickers, period=period, macro_tickers=["^VIX", "^TNX", "GLD", "XLK", "XLP"]
).load()

[*********************100%***********************]  16 of 16 completed
/home/martijn_d_jansen/reinforcement-learning-trader/src/trader/stock_data_loader.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.data[("Global", "Regime_XLK_XLP")] = (
/home/martijn_d_jansen/reinforcement-learning-trader/src/trader/stock_data_loader.py:73: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  self.data = self.data.drop(["Open", "Close", "High", "Low", "Volume"], axis=1)


In [ ]:
%reload_ext autoreload


import numpy as np
from stable_baselines3 import PPO

from trader import DirichletPolicy, LivePlotCallback, TraderEnvironment

total_timesteps = 1000000

env_config = {
    "budget": np.float32(10000.0),
    "episode_length": 252,
    "history_length": 30,
    "fee_rate": np.float32(0.0005),
    "lmb": np.float32(1),
}


train_env = TraderEnvironment(env_data=train_env_data, **env_config)

val_env = TraderEnvironment(env_data=val_env_data, **env_config)
callback = LivePlotCallback(
    total_timesteps=total_timesteps,
    train_env=train_env,
    val_env=val_env,
    tickers=tickers,
)

model = PPO(
    DirichletPolicy,
    train_env,
    verbose=0,
    device="cuda",
)

trained_model = model.learn(
    total_timesteps=total_timesteps, log_interval=4, callback=callback
)

In [ ]:
%reload_ext autoreload


import numpy as np
from sb3_contrib import TQC

from trader import LivePlotCallback, TraderEnvironment

total_timesteps = 1000000

env_config = {
    "budget": np.float32(10000.0),
    "episode_length": 252,
    "history_length": 30,
    "fee_rate": np.float32(0.0005),
    "alpha": np.float32(0.5),
    "distribution": "gaussian",
}


train_env = TraderEnvironment(env_data=train_env_data, **env_config)

val_env = TraderEnvironment(env_data=val_env_data, **env_config)
callback = LivePlotCallback(
    total_timesteps=total_timesteps,
    train_env=train_env,
    val_env=val_env,
    tickers=tickers,
)

model = TQC(
    "MlpPolicy",
    train_env,
    top_quantiles_to_drop_per_net=2,
    verbose=0,
    device="cuda",
)

trained_model = model.learn(
    total_timesteps=total_timesteps, log_interval=4, callback=callback
)